# Imports

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import pandas as pd
import numpy as np
import os


# Load data

In [ ]:
maison_df = pd.read_csv('datasets_prepd/dvf_maison.csv')
print(maison_df.head())

# Features

In [ ]:
# Select relevant features for Maison and the target (valeur_fonciere_actualisee)
maison_features = maison_df[[
    "longitude", "latitude", "code_postal", "surface_reelle_bati", 
    "nombre_pieces_principales", "prix_m2_ref", "surface_terrain", "number_of_lots"
]]

# Target: Actualized sale price
maison_target = maison_df["valeur_fonciere_actualisee"]

# Inspect the first few rows to verify the selected features
print(maison_features.head())
print(maison_target.head())

# Make sets : trail and test and validate

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into train, validation, and test sets (80% train, 10% validation, 10% test)
X_train, X_temp, y_train, y_temp = train_test_split(maison_features, maison_target, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Inspect the sizes of the splits
print(f"Train set: {len(X_train)}, Validation set: {len(X_val)}, Test set: {len(X_test)}")

# Ssave

In [ ]:
os.makedirs('data_maison', exist_ok=True)  # Create 'data_maison' directory if it doesn't exist

# Save the features and target for each set to CSV for later model training
X_train["valeur_fonciere_actualisee"] = y_train
X_val["valeur_fonciere_actualisee"] = y_val
X_test["valeur_fonciere_actualisee"] = y_test

X_train.to_csv('data_maison/maison_train.csv', index=False)
X_val.to_csv('data_maison/maison_val.csv', index=False)
X_test.to_csv('data_maison/maison_test.csv', index=False)

Train model

In [ ]:
# Load the preprocessed Maison training data
maison_train = pd.read_csv('data_maison/maison_train.csv')

# Separate features (X) and target (y)
X_train_maison = maison_train.drop(columns=["valeur_fonciere_actualisee"])  # Features (excluding target)
y_train_maison = maison_train["valeur_fonciere_actualisee"]  # Target variable

# Inspect the first few rows of the features
print(X_train_maison.head())
print(y_train_maison.head())

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Initialize the Random Forest Regressor model
model_maison = RandomForestRegressor(n_estimators=100, random_state=42)

# Train the model
model_maison.fit(X_train_maison, y_train_maison)

# Make predictions on the training data
y_train_pred_maison = model_maison.predict(X_train_maison)

# Evaluation

In [ ]:
# Calculate MSE
mse_maison = mean_squared_error(y_train_maison, y_train_pred_maison)

# Calculate RMSE manually by taking the square root of MSE
rmse_maison = np.sqrt(mse_maison)

# Now calculate MAE and R² as usual
mae_maison = mean_absolute_error(y_train_maison, y_train_pred_maison)
r2_maison = r2_score(y_train_maison, y_train_pred_maison)

# Print out the performance metrics
print(f"Maison Model Training Performance:")
print(f"RMSE: {rmse_maison}")
print(f"MAE: {mae_maison}")
print(f"R²: {r2_maison}")

Save model

In [ ]:
import joblib #import now so we have more memory for training the model
import os

os.makedirs('models', exist_ok=True)  # Create 'models' directory if it doesn't exist

# Save the trained Maison model
joblib.dump(model_maison, 'models/maison_random_forest_model.pkl')

print("Maison model saved to 'models/maison_random_forest_model.pkl'")